In [2]:
import os
import time
import subprocess
import numpy as np
import xml.etree.ElementTree as ET
from pynq import Overlay, MMIO, allocate

BITFILE   = "WNNAcceleratorBlk.bit" 
DATA_FILE = "mnist_fpga_test_data.npz"
LUT_DIR   = "luts"
CPP_EXE   = "wnn_runner"

NUM_LUTS         = 500
N_CLASSES        = 10
ADDR_BITS        = 6
M                = 1 << ADDR_BITS 
LUT_DATA_WIDTH   = 8      
WORDS_PER_ENTRY  = 4      
INPUT_BITS       = 25088 
DMA_TRANSFER_LEN = INPUT_BITS // 32 

# Load Hardware
if not os.path.exists(BITFILE): raise FileNotFoundError(f"Missing {BITFILE}")
overlay = Overlay(BITFILE) 

try:
    dma = overlay.axi_dma_0
    wnn_ip = overlay.wnn_axi_0 
except AttributeError:
    # Fallback: Search by IP type if standard names fail
    for name, ip in overlay.ip_dict.items():
        if 'wnn' in name.lower(): wnn_ip = MMIO(ip['phys_addr'], ip['addr_range'])
        if 'dma' in name.lower(): dma = getattr(overlay, name)
if not wnn_ip or not dma: raise Exception("IPs not found!")

def get_bram_driver(overlay, bitfile_path):
    keys = [k for k in overlay.ip_dict.keys() if 'bram' in k.lower()]
    if keys: return overlay.ip_dict[keys[0]], overlay.ip_dict[keys[0]].mmio.length
    
    # Fallback: Parse HWH file for base address
    hwh_path = bitfile_path.replace(".bit", ".hwh")
    if not os.path.exists(hwh_path): return None, 0
    tree = ET.parse(hwh_path)
    for module in tree.getroot().iter('MODULE'):
        if 'axi_bram_ctrl' in module.get('VLNV', '').lower():
            base = int(next(p.get('VALUE') for p in module.iter('PARAMETER') if p.get('NAME') == 'C_S_AXI_BASEADDR'), 16)
            high = int(next(p.get('VALUE') for p in module.iter('PARAMETER') if p.get('NAME') == 'C_S_AXI_HIGHADDR'), 16)
            return MMIO(base, high-base+1), high-base+1
    raise Exception("BRAM not found")

bram_ip, bram_size = get_bram_driver(overlay, BITFILE)

# Program Weights
print(f"Programming Weights...")
MAX_VAL = (1 << LUT_DATA_WIDTH) - 1 
for l in range(NUM_LUTS):
    mem_path = os.path.join(LUT_DIR, f"lut_{l:03d}.mem")
    with open(mem_path, 'r') as f: lines = f.readlines()
    
    for addr_idx, line in enumerate(lines):
        if addr_idx >= M: break
        counts = [int(x, 16) for x in line.strip().split()] 
        
        # Pack 10 classes into 128 bits
        packed_val = 0
        for c in range(N_CLASSES):
            val = min(counts[c], MAX_VAL)
            packed_val |= (val << (c * LUT_DATA_WIDTH))
            
        entry_offset = (l * M + addr_idx) * (WORDS_PER_ENTRY * 4)
        for w in range(WORDS_PER_ENTRY):
            chunk = (packed_val >> (w * 32)) & 0xFFFFFFFF
            bram_ip.write(entry_offset + (w*4), chunk)

# Data Prep
print("Packing Data...")
data = np.load(DATA_FILE)
x_test_bits = data['x'] 
y_test = data['y']      
total_images = len(y_test)

# Allocate contiguous memory for C++ access
big_buffer = allocate(shape=(total_images * DMA_TRANSFER_LEN,), dtype=np.uint32)

t_pack = time.time()
for i in range(total_images):
    packed_bytes = np.packbits(x_test_bits[i].astype(np.uint8), bitorder='little')
    # View as UINT32 to ensure correct endianness before writing to CMA
    packed_words = np.frombuffer(packed_bytes, dtype=np.uint32)
    
    start_idx = i * DMA_TRANSFER_LEN
    big_buffer[start_idx : start_idx + DMA_TRANSFER_LEN] = packed_words

big_buffer.flush() 
y_test.astype(np.uint8).tofile("y_test.bin")
print(f"Packing complete in {time.time() - t_pack:.2f}s")

# Execute
print("\nLaunching C++ Accelerator...")
cmd = [
    f"./{CPP_EXE}", 
    str(wnn_ip.mmio.base_addr), 
    str(dma.mmio.base_addr), 
    str(big_buffer.device_address), 
    str(total_images),
    "y_test.bin"
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
out, err = proc.communicate()
print(out)
if err: print("STDERR:", err)

del big_buffer

Programming Weights...
Packing Data...
Packing complete in 9.39s

Launching C++ Accelerator...
[C++] Starting Maximum Stable Speed Loop...

[C++] FINAL REPORT:
      Accuracy: 95.64% (9564/10000)
      Time:     0.6567 s
      FPS:      15226.77

